In [1]:
import csv
import re
import time
import xml.etree.ElementTree as ET
from collections import deque
from urllib.parse import urljoin, urlparse, urldefrag
from urllib.robotparser import RobotFileParser

import requests
from bs4 import BeautifulSoup

INPUT_FILE = r"C:\Users\ASUS\Downloads/companies.csv"
OUTPUT_FILE = r"C:\Users\ASUS\Downloads/pune_public_contacts.csv"

MAX_PAGES_PER_COMPANY = 50
REQUEST_DELAY = 1.5
TIMEOUT = 20

USER_AGENT = "PublicBusinessContactResearchBot/1.0"

HEADERS = {
    "User-Agent": USER_AGENT,
    "Accept": "text/html,application/xhtml+xml,application/xml"
}

ROLE_KEYWORDS = [
    "human resources",
    "human resource",
    "hr manager",
    "hr head",
    "head hr",
    "people officer",
    "people operations",
    "employee engagement",
    "employee wellness",
    "corporate wellness",
    "occupational health",
    "administration manager",
    "admin manager",
    "facilities manager",
    "facility manager",
    "ehs manager",
    "health and safety",
    "csr head",
    "corporate social responsibility",
    "talent acquisition",
    "talent management",
    "leadership team",
]

URL_KEYWORDS = [
    "contact",
    "contact-us",
    "leadership",
    "management",
    "our-team",
    "team",
    "people",
    "human-resource",
    "human-resources",
    "employee",
    "wellness",
    "occupational",
    "health",
    "safety",
    "ehs",
    "csr",
    "administration",
    "facilities",
    "pune",
    "india",
    "office",
    "location",
]

PUNE_KEYWORDS = [
    "pune",
    "hinjawadi",
    "hinjewadi",
    "kharadi",
    "hadapsar",
    "magarpatta",
    "baner",
    "balewadi",
    "wakad",
    "pimpri",
    "chinchwad",
    "chakan",
    "talawade",
    "viman nagar",
    "kalyani nagar",
    "yerawada",
    "senapati bapat",
    "mundhwa",
    "shivajinagar",
    "kothrud",
]

EMAIL_RE = re.compile(
    r"\b[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}\b"
)

PHONE_RE = re.compile(
    r"(?<!\d)"
    r"(?:\+91[\s().\-]*)?"
    r"(?:"
    r"[6-9]\d{4}[\s.\-]?\d{5}"
    r"|0?20[\s().\-]*\d{3,4}[\s.\-]*\d{4}"
    r")"
    r"(?!\d)"
)

NAME_ROLE_RE = re.compile(
    r"([A-Z][A-Za-z.'\-]+(?:\s+[A-Z][A-Za-z.'\-]+){1,3})"
    r"\s*[-–—|,:]\s*"
    r"("
    r"(?:Chief\s+)?Human Resources(?:\s+Officer)?|"
    r"HR\s+(?:Head|Manager|Director)|"
    r"Head\s*[-–—]?\s*HR|"
    r"People\s+(?:Officer|Director|Manager)|"
    r"Administration\s+Manager|"
    r"Facilities\s+Manager|"
    r"EHS\s+Manager|"
    r"CSR\s+Head"
    r")",
    re.IGNORECASE
)

EXCLUDED_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".gif", ".svg", ".webp",
    ".zip", ".mp4", ".mp3", ".avi", ".mov",
    ".doc", ".docx", ".xls", ".xlsx", ".ppt", ".pptx"
)

BAD_EMAIL_ENDINGS = (
    ".png", ".jpg", ".jpeg", ".gif", ".svg", ".webp",
    ".css", ".js"
)


def normalize_domain(domain):
    domain = domain.strip().lower()
    domain = domain.replace("https://", "").replace("http://", "")
    domain = domain.split("/")[0]
    return domain.removeprefix("www.")


def canonical_url(url):
    url = urldefrag(url)[0]
    parsed = urlparse(url)

    if parsed.scheme not in ("http", "https"):
        return None

    return parsed._replace(query="").geturl().rstrip("/")


def same_domain(url, company_domain):
    host = urlparse(url).netloc.lower().split(":")[0]
    host = host.removeprefix("www.")
    company_domain = normalize_domain(company_domain)

    return (
        host == company_domain
        or host.endswith("." + company_domain)
    )


def likely_html_url(url):
    path = urlparse(url).path.lower()
    return not path.endswith(EXCLUDED_EXTENSIONS)


class RobotsManager:
    def __init__(self):
        self.cache = {}

    def parser_for(self, url):
        parsed = urlparse(url)
        base = f"{parsed.scheme}://{parsed.netloc}"

        if base in self.cache:
            return self.cache[base]

        robots_url = urljoin(base, "/robots.txt")
        parser = RobotFileParser()
        parser.set_url(robots_url)

        try:
            response = requests.get(
                robots_url,
                headers=HEADERS,
                timeout=TIMEOUT
            )

            if response.status_code == 200:
                parser.parse(response.text.splitlines())
            elif response.status_code in (401, 403):
                parser.parse(["User-agent: *", "Disallow: /"])
            else:
                parser.parse(["User-agent: *", "Disallow:"])
        except requests.RequestException:
            parser.parse(["User-agent: *", "Disallow:"])

        self.cache[base] = parser
        return parser

    def allowed(self, url):
        try:
            return self.parser_for(url).can_fetch(USER_AGENT, url)
        except Exception:
            return False

    def sitemap_urls(self, url):
        parser = self.parser_for(url)

        try:
            return parser.site_maps() or []
        except Exception:
            return []


robots = RobotsManager()


def download(url):
    if not robots.allowed(url):
        return None

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=TIMEOUT,
            allow_redirects=True
        )

        if response.status_code != 200:
            return None

        return response

    except requests.RequestException:
        return None


def extract_sitemap_locations(xml_text):
    try:
        root = ET.fromstring(xml_text)
    except ET.ParseError:
        return []

    locations = []

    for element in root.iter():
        if element.tag.lower().endswith("loc") and element.text:
            locations.append(element.text.strip())

    return locations


def get_sitemap_pages(homepage, domain):
    parsed = urlparse(homepage)
    base = f"{parsed.scheme}://{parsed.netloc}"

    sitemap_candidates = set(robots.sitemap_urls(homepage))
    sitemap_candidates.update({
        urljoin(base, "/sitemap.xml"),
        urljoin(base, "/sitemap_index.xml"),
        urljoin(base, "/sitemap-index.xml"),
        urljoin(base, "/wp-sitemap.xml"),
    })

    sitemap_queue = deque((url, 0) for url in sitemap_candidates)
    checked_sitemaps = set()
    page_urls = set()

    while sitemap_queue:
        sitemap_url, depth = sitemap_queue.popleft()

        if sitemap_url in checked_sitemaps or depth > 2:
            continue

        checked_sitemaps.add(sitemap_url)
        response = download(sitemap_url)

        if not response:
            continue

        content_type = response.headers.get(
            "Content-Type", ""
        ).lower()

        if (
            "xml" not in content_type
            and not sitemap_url.lower().endswith(".xml")
        ):
            continue

        locations = extract_sitemap_locations(response.text)

        for location in locations:
            if not same_domain(location, domain):
                continue

            lower_location = location.lower()

            if (
                "sitemap" in lower_location
                or lower_location.endswith(".xml")
            ):
                sitemap_queue.append((location, depth + 1))
            elif likely_html_url(location):
                page_urls.add(canonical_url(location))

    return {url for url in page_urls if url}


def page_score(url):
    lower_url = url.lower()
    score = 0

    for keyword in URL_KEYWORDS:
        if keyword in lower_url:
            score += 3

    if "pune" in lower_url:
        score += 5

    if any(term in lower_url for term in ["career", "jobs", "vacancy"]):
        score -= 1

    if any(term in lower_url for term in [
        "privacy", "terms", "cookie", "investor",
        "annual-report", "press-release", "blog"
    ]):
        score -= 4

    return score


def choose_sitemap_pages(urls):
    scored = sorted(
        urls,
        key=lambda url: (page_score(url), url),
        reverse=True
    )

    relevant = [
        url for url in scored
        if page_score(url) > 0
    ]

    return relevant[:MAX_PAGES_PER_COMPANY]


def deobfuscate(text):
    substitutions = [
        (r"\s*\[\s*at\s*\]\s*", "@"),
        (r"\s*\(\s*at\s*\)\s*", "@"),
        (r"\s+at\s+", "@"),
        (r"\s*\[\s*dot\s*\]\s*", "."),
        (r"\s*\(\s*dot\s*\)\s*", "."),
        (r"\s+dot\s+", "."),
    ]

    output = text

    for pattern, replacement in substitutions:
        output = re.sub(
            pattern,
            replacement,
            output,
            flags=re.IGNORECASE
        )

    return output


def clean_email(email):
    return email.strip(".,;:()[]<>").lower()


def valid_email(email):
    if email.endswith(BAD_EMAIL_ENDINGS):
        return False

    if "example.com" in email:
        return False

    return bool(EMAIL_RE.fullmatch(email))


def clean_phone(phone):
    return re.sub(r"\s+", " ", phone).strip(" .,-()")


def extract_contact_context(element):
    text = element.get_text(" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    parent = element.parent

    for _ in range(2):
        if parent is None:
            break

        parent_text = re.sub(
            r"\s+",
            " ",
            parent.get_text(" ", strip=True)
        )

        if len(parent_text) <= 600:
            text = parent_text
            break

        parent = parent.parent

    return text[:600]


def detect_role(text):
    lower_text = text.lower()

    found = [
        keyword
        for keyword in ROLE_KEYWORDS
        if keyword in lower_text
    ]

    return "; ".join(found)


def detect_pune(text):
    lower_text = text.lower()

    found = [
        location
        for location in PUNE_KEYWORDS
        if location in lower_text
    ]

    return "; ".join(found)


def detect_name_role(text):
    match = NAME_ROLE_RE.search(text)

    if not match:
        return "", ""

    return match.group(1).strip(), match.group(2).strip()


def extract_page_contacts(url, company):
    response = download(url)

    if not response:
        return [], set()

    content_type = response.headers.get(
        "Content-Type", ""
    ).lower()

    if "text/html" not in content_type:
        return [], set()

    soup = BeautifulSoup(response.text, "html.parser")

    for tag in soup([
        "script", "style", "noscript", "svg", "template"
    ]):
        tag.decompose()

    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    page_text = soup.get_text(" ", strip=True)
    page_text = re.sub(r"\s+", " ", page_text)

    page_role = detect_role(page_text)
    page_location = detect_pune(page_text)

    contacts = []
    discovered_links = set()
    seen_records = set()

    for link in soup.select("a[href]"):
        href = link.get("href", "").strip()
        absolute = canonical_url(urljoin(url, href))

        if (
            absolute
            and same_domain(absolute, urlparse(url).netloc)
            and likely_html_url(absolute)
        ):
            combined = (
                absolute + " " + link.get_text(" ", strip=True)
            ).lower()

            if any(keyword in combined for keyword in URL_KEYWORDS):
                discovered_links.add(absolute)

        if href.lower().startswith("mailto:"):
            raw_email = href[7:].split("?")[0]
            email = clean_email(raw_email)

            if valid_email(email):
                context = extract_contact_context(link)
                name, specific_role = detect_name_role(context)

                key = ("email", email, context)

                if key not in seen_records:
                    seen_records.add(key)
                    contacts.append({
                        "company": company,
                        "name": name,
                        "role": specific_role or detect_role(context) or page_role,
                        "pune_context": detect_pune(context) or page_location,
                        "email": email,
                        "phone": "",
                        "context": context,
                        "page_title": title,
                        "source_url": url,
                    })

        elif href.lower().startswith("tel:"):
            phone = clean_phone(href[4:])
            context = extract_contact_context(link)
            name, specific_role = detect_name_role(context)

            key = ("phone", phone, context)

            if phone and key not in seen_records:
                seen_records.add(key)
                contacts.append({
                    "company": company,
                    "name": name,
                    "role": specific_role or detect_role(context) or page_role,
                    "pune_context": detect_pune(context) or page_location,
                    "email": "",
                    "phone": phone,
                    "context": context,
                    "page_title": title,
                    "source_url": url,
                })

    searchable_text = deobfuscate(page_text)

    for email_match in EMAIL_RE.finditer(searchable_text):
        email = clean_email(email_match.group())

        if not valid_email(email):
            continue

        start = max(0, email_match.start() - 250)
        end = min(len(searchable_text), email_match.end() + 250)
        context = searchable_text[start:end]
        name, specific_role = detect_name_role(context)

        key = ("email", email, context)

        if key not in seen_records:
            seen_records.add(key)
            contacts.append({
                "company": company,
                "name": name,
                "role": specific_role or detect_role(context) or page_role,
                "pune_context": detect_pune(context) or page_location,
                "email": email,
                "phone": "",
                "context": context,
                "page_title": title,
                "source_url": url,
            })

    for phone_match in PHONE_RE.finditer(page_text):
        phone = clean_phone(phone_match.group())
        start = max(0, phone_match.start() - 250)
        end = min(len(page_text), phone_match.end() + 250)
        context = page_text[start:end]
        name, specific_role = detect_name_role(context)

        key = ("phone", phone, context)

        if key not in seen_records:
            seen_records.add(key)
            contacts.append({
                "company": company,
                "name": name,
                "role": specific_role or detect_role(context) or page_role,
                "pune_context": detect_pune(context) or page_location,
                "email": "",
                "phone": phone,
                "context": context,
                "page_title": title,
                "source_url": url,
            })

    return contacts, discovered_links


def find_working_homepage(domain):
    candidates = [
        f"https://{domain}",
        f"https://www.{domain}",
        f"http://{domain}",
        f"http://www.{domain}",
    ]

    for candidate in candidates:
        response = download(candidate)

        if response and "text/html" in response.headers.get(
            "Content-Type", ""
        ).lower():
            return response.url.rstrip("/")

    return None


def crawl_company(company, domain):
    domain = normalize_domain(domain)
    homepage = find_working_homepage(domain)

    if not homepage:
        print(f"Could not access website for {company}")
        return []

    print(f"Homepage: {homepage}")

    sitemap_pages = get_sitemap_pages(homepage, domain)
    selected_pages = choose_sitemap_pages(sitemap_pages)

    seed_urls = [
        homepage,
        urljoin(homepage + "/", "contact"),
        urljoin(homepage + "/", "contact-us"),
        urljoin(homepage + "/", "about-us"),
        urljoin(homepage + "/", "leadership"),
        urljoin(homepage + "/", "our-team"),
        urljoin(homepage + "/", "csr"),
        urljoin(homepage + "/", "locations"),
        urljoin(homepage + "/", "india"),
    ]

    queue = deque()
    queued = set()

    for url in seed_urls + selected_pages:
        normalized = canonical_url(url)

        if (
            normalized
            and normalized not in queued
            and same_domain(normalized, domain)
        ):
            queue.append(normalized)
            queued.add(normalized)

    visited = set()
    all_contacts = []

    while queue and len(visited) < MAX_PAGES_PER_COMPANY:
        url = queue.popleft()

        if url in visited:
            continue

        visited.add(url)
        print(f"  Checking {url}")

        contacts, links = extract_page_contacts(url, company)
        all_contacts.extend(contacts)

        ranked_links = sorted(
            links,
            key=page_score,
            reverse=True
        )

        for link in ranked_links:
            if (
                link not in visited
                and link not in queued
                and page_score(link) > 0
            ):
                queue.append(link)
                queued.add(link)

        time.sleep(REQUEST_DELAY)

    return all_contacts


def contact_priority(row):
    score = 0

    if row["role"]:
        score += 5

    if row["pune_context"]:
        score += 5

    if row["name"]:
        score += 3

    if row["email"]:
        score += 2

    if row["phone"]:
        score += 1

    return score


def main():
    all_rows = []

    with open(
        INPUT_FILE,
        newline="",
        encoding="utf-8-sig"
    ) as file:
        reader = csv.DictReader(file)

        for item in reader:
            company = item["company"].strip()
            domain = item["domain"].strip()

            print(f"\nProcessing {company}")
            all_rows.extend(crawl_company(company, domain))

    unique = {}

    for row in all_rows:
        key = (
            row["company"].lower(),
            row["email"].lower(),
            row["phone"],
            row["source_url"],
        )

        existing = unique.get(key)

        if existing is None or contact_priority(row) > contact_priority(existing):
            unique[key] = row

    final_rows = sorted(
        unique.values(),
        key=lambda row: (
            row["company"].lower(),
            -contact_priority(row),
            row["source_url"]
        )
    )

    fields = [
        "company",
        "name",
        "role",
        "pune_context",
        "email",
        "phone",
        "context",
        "page_title",
        "source_url",
    ]

    with open(
        OUTPUT_FILE,
        "w",
        newline="",
        encoding="utf-8-sig"
    ) as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(final_rows)

    print(
        f"\nSaved {len(final_rows)} public records "
        f"to {OUTPUT_FILE}"
    )


if __name__ == "__main__":
    main()


Processing Persistent Systems
Homepage: https://www.persistent.com
  Checking https://www.persistent.com
  Checking https://www.persistent.com/contact
  Checking https://www.persistent.com/contact-us
  Checking https://www.persistent.com/about-us
  Checking https://www.persistent.com/leadership
  Checking https://www.persistent.com/our-team
  Checking https://www.persistent.com/csr
  Checking https://www.persistent.com/locations
  Checking https://www.persistent.com/india
  Checking https://www.persistent.com/client-success/leading-indian-health-insurance-provider-delivers-personalized-wellness-programs
  Checking https://www.persistent.com/media/press-releases/persistent-systems-co-organizes-first-tcga-conference-and-workshop-in-india-in-collaboration-with-pccm-and-iiser-pune
  Checking https://www.persistent.com/services/cx-transformation/future-driven-innovations-healthcare-provider-network-and-relationship-management
  Checking https://www.persistent.com/industries/healthcare/heal